In [16]:
import pandas as pd


In [17]:
import os
os.listdir("../data")

['dim_customer.csv',
 'dim_delivery_partner_.csv',
 'dim_menu_item.csv',
 'dim_restaurant.csv',
 'fact_delivery_performance.csv',
 'fact_orders.csv',
 'fact_order_items.csv',
 'fact_ratings.csv']

In [18]:
customers =pd.read_csv("../data/dim_customer.csv")
delivery_partners = pd.read_csv("../data/dim_delivery_partner_.csv")
menu_items = pd.read_csv("../data/dim_menu_item.csv")
restaurants = pd.read_csv("../data/dim_restaurant.csv")

order = pd.read_csv("../data/fact_orders.csv")
delivery = pd.read_csv('../data/fact_delivery_performance.csv')
ratings = pd.read_csv('../data/fact_ratings.csv')

# Merge all the main tables 
first join orders + delivery performance 

In [19]:
orders_delivery = order.merge(delivery, on="order_id", how="left")

now we are joiing rating with that 

In [20]:
orders_delivery_ratings = orders_delivery.merge(ratings, on="order_id", how="left")

Now we are adding customer information

In [27]:
print(orders_delivery_ratings.columns.tolist)
print(customers.columns.tolist)

<bound method IndexOpsMixin.tolist of Index(['order_id', 'customer_id', 'restaurant_id_x', 'delivery_partner_id',
       'order_timestamp', 'subtotal_amount', 'discount_amount', 'delivery_fee',
       'total_amount', 'is_cod', 'is_cancelled', 'actual_delivery_time_mins',
       'expected_delivery_time_mins', 'distance_km', 'rating', 'review_text',
       'review_timestamp', 'sentiment_score'],
      dtype='object')>
<bound method IndexOpsMixin.tolist of Index(['customer_id', 'signup_date', 'city', 'acquisition_channel',
       'Unnamed: 4'],
      dtype='object')>


In [31]:
#orders_delivery_ratings = orders_delivery_ratings.drop(columns=["customer_id_y","restaurant_id_y"])
orders_delivery_ratings = orders_delivery_ratings.rename(columns={
    "customer_id_x":"customer_id",
    "restaurant_id_x":"restaurant_id"
})

In [32]:
data = orders_delivery_ratings.merge(customers, on="customer_id", how="left")

In [33]:
print(data.columns.tolist())

['order_id', 'customer_id', 'restaurant_id', 'delivery_partner_id', 'order_timestamp', 'subtotal_amount', 'discount_amount', 'delivery_fee', 'total_amount', 'is_cod', 'is_cancelled', 'actual_delivery_time_mins', 'expected_delivery_time_mins', 'distance_km', 'rating', 'review_text', 'review_timestamp', 'sentiment_score', 'signup_date', 'city', 'acquisition_channel', 'Unnamed: 4']


Now add restaurant information 

In [34]:
data = data.merge(restaurants, on="restaurant_id", how="left")

Add delivery partner info 

In [37]:
data = data.merge(delivery_partners, on="delivery_partner_id", how="left")

# Now we have one final dataset for analysis 
create crisis phase column

In [38]:
data['order_timestamp'] = pd.to_datetime(data['order_timestamp'])

data['phase'] = data['order_timestamp'].apply(lambda x: 'Pre-Crisis' if x.month <= 5 else 'Crisis')

Let's Prepare final dataset property

In [39]:
data['order_timestamp'] = pd.to_datetime(data['order_timestamp'])
data['month'] = data['order_timestamp'].dt.to_period('M').astype(str)
data['year'] = data['order_timestamp'].dt.year
data['month_num'] = data['order_timestamp'].dt.month

Fix the phase column a little better

In [41]:
data['phase'] = data['month_num'].apply(
    lambda x: 'Pre-Crisis' if x <= 5 else 'Crisis'
)

# Now data cleaning first 

In [43]:
data.shape

(149166, 38)

In [44]:
data.isnull().sum()

order_id                            0
customer_id                         0
restaurant_id                       0
delivery_partner_id              5635
order_timestamp                     0
subtotal_amount                     0
discount_amount                     0
delivery_fee                        0
total_amount                        0
is_cod                              0
is_cancelled                        0
actual_delivery_time_mins           0
expected_delivery_time_mins         0
distance_km                         0
rating                          80341
review_text                     80341
review_timestamp                80341
sentiment_score                 80341
signup_date                      5053
city_x                           5053
acquisition_channel              5053
Unnamed: 4                     149166
restaurant_name                     0
city_y                              0
cuisine_type                        0
partner_type                        0
avg_prep_tim

In [46]:
(data.isnull().sum() / len(data)) * 100

order_id                         0.000000
customer_id                      0.000000
restaurant_id                    0.000000
delivery_partner_id              3.777671
order_timestamp                  0.000000
subtotal_amount                  0.000000
discount_amount                  0.000000
delivery_fee                     0.000000
total_amount                     0.000000
is_cod                           0.000000
is_cancelled                     0.000000
actual_delivery_time_mins        0.000000
expected_delivery_time_mins      0.000000
distance_km                      0.000000
rating                          53.860129
review_text                     53.860129
review_timestamp                53.860129
sentiment_score                 53.860129
signup_date                      3.387501
city_x                           3.387501
acquisition_channel              3.387501
Unnamed: 4                     100.000000
restaurant_name                  0.000000
city_y                           0

In [47]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149166 entries, 0 to 149165
Data columns (total 38 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   order_id                     149166 non-null  object        
 1   customer_id                  149166 non-null  object        
 2   restaurant_id                149166 non-null  object        
 3   delivery_partner_id          143531 non-null  object        
 4   order_timestamp              149166 non-null  datetime64[ns]
 5   subtotal_amount              149166 non-null  float64       
 6   discount_amount              149166 non-null  float64       
 7   delivery_fee                 149166 non-null  float64       
 8   total_amount                 149166 non-null  float64       
 9   is_cod                       149166 non-null  object        
 10  is_cancelled                 149166 non-null  object        
 11  actual_delivery_time_mins 

In [48]:
data.duplicated().sum()

np.int64(0)

In [49]:
data = data.drop_duplicates()

Proper cleaning steps we should follow
Step 1 — Drop fully useless columns

In [50]:
data = data.drop(columns=['Unnamed: 4'], errors='ignore')

Step 2 — Rename confusing columns

In [51]:
data = data.rename(columns={
    'city_x':'customer_city',
    'city_y': 'restaurants_city',
    'city' : 'partner_city',
    'is_active_x' : 'restaurant_is_active',
    'is_active_y' : 'partner_is_active'
})

In [52]:
object_cols = data.select_dtypes(include='object').columns

for col in object_cols:
    print(f"\nColumn: {col}")
    print(data[col].dropna().unique()[:10])


Column: order_id
['ORD202501023439' 'ORD202501012051' 'ORD202501019281' 'ORD202501000124'
 'ORD202501006518' 'ORD202501018255' 'ORD202501004299' 'ORD202501018036'
 'ORD202501009329' 'ORD202501007498']

Column: customer_id
['CUST181110' 'CUST025572' 'CUST179306' 'CUST191820' 'CUST033760'
 'CUST011850' 'CUST107475' 'CUST093042' 'CUST104825' 'CUST135654']

Column: restaurant_id
['REST08622' 'REST02383' 'REST14069' 'REST19745' 'REST12962' 'REST01307'
 'REST12542' 'REST13907' 'REST10267' 'REST05434']

Column: delivery_partner_id
['DP05541' 'DP08091' 'DP02021' 'DP13859' 'DP09615' 'DP14063' 'DP07728'
 'DP01276' 'DP03078' 'DP11625']

Column: is_cod
['N' 'Y']

Column: is_cancelled
['N' 'Y']

Column: review_text
['Super fast delivery' 'Great taste!' 'Tasty but a bit late'
 'Satisfied overall' 'Loved it!' 'Excellent service' 'Fresh and delicious'
 'Could be hotter' 'Good but can improve' 'Okay experience']

Column: review_timestamp
['01-01-2025 15:00' '01-01-2025 14:03' '01-01-2025 14:06'
 '01-0

Step 4 — Fix Boolean Columns

In [54]:
clean_data = data.copy()

In [56]:
clean_data['is_cod'] = clean_data['is_cod'].map({'Y':1,'N':0})
clean_data['is_cancelled'] = clean_data['is_cancelled'].map({'Y':1,'N':0})
clean_data['restaurant_is_active'] = clean_data['restaurant_is_active'].map({'Y':1,'N':0})
clean_data['partner_is_active'] = clean_data['partner_is_active'].map({'Y':1,'N':0})

In [57]:
clean_data['signup_data'] = pd.to_datetime(clean_data['signup_date'],dayfirst=True, errors='coerce')
clean_data['review_timestamp'] = pd.to_datetime(clean_data['review_timestamp'], dayfirst=True,errors='coerce')

In [58]:
clean_data['review_text'] = clean_data['review_text'].fillna('No Review')
clean_data['rating'] = clean_data['rating'].fillna(0)
clean_data['sentiment_score'] = clean_data['sentiment_score'].fillna(0)

In [59]:
clean_data['customer_city'] = clean_data['customer_city'].fillna('Unknown')
clean_data['acquisition_channel'] = clean_data['acquisition_channel'].fillna('Unknown')

In [60]:
clean_data['partner_name'] = clean_data['partner_name'].fillna('Unknown')
clean_data['partner_city'] = clean_data['partner_city'].fillna('Unknown')
clean_data['vehicle_type'] = clean_data['vehicle_type'].fillna('Unknown')
clean_data['employment_type'] = clean_data['employment_type'].fillna('Unknown')
clean_data['avg_rating'] = clean_data['avg_rating'].fillna(0)
clean_data['partner_is_active'] = clean_data['partner_is_active'].fillna(0)

In [61]:
clean_data['avg_prep_time_min'] = clean_data['avg_prep_time_min'].astype('category')

In [62]:
clean_data = clean_data.drop(columns=['restaurant_name'])

In [63]:
clean_data.isnull().sum()

order_id                            0
customer_id                         0
restaurant_id                       0
delivery_partner_id              5635
order_timestamp                     0
subtotal_amount                     0
discount_amount                     0
delivery_fee                        0
total_amount                        0
is_cod                         149166
is_cancelled                        0
actual_delivery_time_mins           0
expected_delivery_time_mins         0
distance_km                         0
rating                              0
review_text                         0
review_timestamp                80341
sentiment_score                     0
signup_date                      5053
customer_city                       0
acquisition_channel                 0
restaurants_city                    0
cuisine_type                        0
partner_type                        0
avg_prep_time_min                   0
restaurant_is_active                0
partner_name

In [64]:
clean_data.duplicated().sum()
clean_data['order_id'].duplicated().sum()

np.int64(0)

In [65]:
clean_data['is_cod'] = clean_data['is_cod'].str.strip().map({'Y':1, 'N':0})

AttributeError: Can only use .str accessor with string values!

In [66]:
clean_data['is_cod'].unique()

array([nan])

In [67]:
clean_data['is_cod'] = data['is_cod'].map({'Y':1, 'N':0})

In [68]:
clean_data['is_cod'].value_counts()

is_cod
0    102351
1     46815
Name: count, dtype: int64

In [69]:
clean_data.isnull().sum()

order_id                           0
customer_id                        0
restaurant_id                      0
delivery_partner_id             5635
order_timestamp                    0
subtotal_amount                    0
discount_amount                    0
delivery_fee                       0
total_amount                       0
is_cod                             0
is_cancelled                       0
actual_delivery_time_mins          0
expected_delivery_time_mins        0
distance_km                        0
rating                             0
review_text                        0
review_timestamp               80341
sentiment_score                    0
signup_date                     5053
customer_city                      0
acquisition_channel                0
restaurants_city                   0
cuisine_type                       0
partner_type                       0
avg_prep_time_min                  0
restaurant_is_active               0
partner_name                       0
p

In [70]:
clean_data = clean_data.drop(columns=['signup_data'], errors='ignore')

In [71]:
clean_data.isnull().sum()

order_id                           0
customer_id                        0
restaurant_id                      0
delivery_partner_id             5635
order_timestamp                    0
subtotal_amount                    0
discount_amount                    0
delivery_fee                       0
total_amount                       0
is_cod                             0
is_cancelled                       0
actual_delivery_time_mins          0
expected_delivery_time_mins        0
distance_km                        0
rating                             0
review_text                        0
review_timestamp               80341
sentiment_score                    0
signup_date                     5053
customer_city                      0
acquisition_channel                0
restaurants_city                   0
cuisine_type                       0
partner_type                       0
avg_prep_time_min                  0
restaurant_is_active               0
partner_name                       0
p

# save the clean dataset

In [72]:
clean_data.to_csv("clean_food_delivery_data.csv", index=False)

# Question 1 : Monthly Order
compare total order across pre-crisis vs crisis 

In [73]:
clean_data[['month','order_id']].groupby('month').count()

,order_id
month,
2025-01,23539
2025-02,22667
2025-03,23543
2025-04,21466
2025-05,22591
2025-06,9293
2025-07,8818
2025-08,8555
2025-09,8694
